# nb06 — реконструкция LaTeX из текстового слоя PDF (кириллица без OCR)

Продолжение `nb05`. Цель — снять главный блокер pix2text: кириллические индексы формул.
Гипотеза: для text-based PDF **текстовый слой уже содержит правильную кириллицу**, а
`pdfplumber` отдаёт подстрочные символы с явной геометрией (`size`, `top`). Значит LaTeX
можно собрать детерминированно, без OCR и моделей.

Проверка — на СП 31.13330.2021 (тот же документ, где pix2text давал 19% → 48%).
Подробности реализации — `research/text_layer_latex.py`; стек и открытые пункты —
`PIPELINE.md`; сравнение ML-подходов — `RECOMMENDATION.md`.

## 0. Окружение

In [ ]:
from pathlib import Path
import re
from collections import Counter

import pdfplumber

from normatives_pipeline.parser import extractors, text_layer
from normatives_pipeline.parser.symbols import normalize_symbol
from normatives_pipeline.formulas.validation import latex2sympy
import research.text_layer_latex as tll

PDF = Path("../docs/normative-base/СП 31.13330.2021.pdf")
print("документ:", PDF.name, "| есть:", PDF.exists())

документ: СП 31.13330.2021.pdf | есть: True


## 1. Текстовый слой хранит кириллицу корректно

Извлекаем формулы регуляркой (как в основном пайплайне) и печатаем те, где в обозначении
есть кириллица. Это `pdftotext -layout`: индексы уплощены в строку, но **буквы верные**.

In [ ]:
pages = text_layer.read_pages(PDF)
forms = extractors.extract_formulas(pages)
cyr_forms = [f for f in forms if re.search(r"[а-яё]", f.raw_text)]
print(f"формул всего: {len(forms)}, с кириллицей: {len(cyr_forms)}")
for f in cyr_forms:
    print(f"  ({f.number}) p{f.page}: {f.raw_text.strip()[:88]}")

формул всего: 13, с кириллицей: 13
  (1) p12: 5.2 Расчетный (средний за год) суточный расход воды Qж, м3/сут, на хозяйственно-питьевые
  (2) p13: степенью благоустройства.     Расчетные расходы воды в сутки наибольшего и наименьшего в
  (3) p13: Kсут. max = 1,1 – 1,3; Kсут. min = 0,7 – 0,9.     Расчетные часовые расходы воды qч, м3/
  (4) p13: q ч. min = K ч. minQ сут. min 24.      Коэффициент часовой неравномерности водопотребл
  (5) p39: Реагенты рекомендуется вводить за 1–3 мин до ввода коагулянтов.     9.16 Дозы подщелачив
  (6) p42: движение воды с поворотами на 180°. Число поворотов следует принимать равным 9–10.     9
  (7) p44: 2 – наибольшей мутности при наибольшем расходе воды, соответствующем этому периоду.     
  (8) p51: 1600 м3/сут должно быть не менее четырех. При производительности станции более 8–10 тыс.
  (9) p51: округлением до ближайших целых чисел (четных или нечетных в зависимости от компоновки фи
  (10) p54: СП 31.13330.2021      9.88 Расстояние от поверхности 

## 2. Геометрия: `size`/`top` отделяют подстрочные символы

`pdfplumber` (в отличие от `extract_text_lines`) отдаёт подстрочный символ как отдельный
глиф: меньший `size` (9 против 14) и больший `top` (ниже базовой линии). Здесь — формула (10)
`h_ж = h_з · a_з / 100 + 0,3` на стр. 54.

In [ ]:
with pdfplumber.open(PDF) as doc:
    page = doc.pages[53]
    for c in page.chars:
        if 84 <= c["top"] <= 100 and c["x0"] > 238 and c["text"].strip():
            kind = "sub " if c["size"] < 12 else "base"
            print(f"  {c['text']!r:>4}  size={c['size']:4.1f}  top={c['top']:6.1f}  {kind}")

   'Н'  size=14.0  top=  91.0  base
   'ж'  size= 9.0  top=  96.5  sub 
   '='  size=14.0  top=  91.0  base
   'Н'  size=14.0  top=  91.0  base
   'з'  size= 9.0  top=  96.5  sub 
   'а'  size=14.0  top=  91.0  base
   'з'  size= 9.0  top=  96.5  sub 
   '/'  size=14.0  top=  91.0  base
   '1'  size=14.0  top=  91.0  base
   '0'  size=14.0  top=  91.0  base
   '0'  size=14.0  top=  91.0  base
   '+'  size=14.0  top=  91.0  base
   '0'  size=14.0  top=  91.0  base
   ','  size=14.0  top=  91.0  base
   '3'  size=14.0  top=  91.0  base
   ','  size=14.0  top=  91.0  base
   '('  size=14.0  top=  91.0  base
   '1'  size=14.0  top=  91.0  base
   '0'  size=14.0  top=  91.0  base
   ')'  size=14.0  top=  91.0  base


## 3. PUA Symbol-шрифта — табличный маппинг

Математические знаки приходят не юникодом, а приватной областью (Adobe Symbol font):
`\uf0e5`=`∑`, `\uf0b3`=`≥`, `\uf0d7`=`⋅`, `\uf0e6..\uf0fb` — растянутые скобки. Это
стандартная кодировка шрифта, а не особенность одного СП, поэтому мапится таблицей.

In [ ]:
with pdfplumber.open(PDF) as doc:
    counter = Counter()
    for page in doc.pages:
        for c in page.chars:
            if 0xE000 <= ord(c["text"]) <= 0xF8FF:
                counter[c["text"]] += 1
print("PUA-глифов:", sum(counter.values()), "| уникальных:", len(counter))
for ch, n in counter.most_common(12):
    mapped = tll.base_char(ch) or "(drop)"
    print(f"  u+{ord(ch):04x} x{n:<3} -> {mapped!r}")

PUA-глифов: 42 | уникальных: 16
  u+f073 x6   -> '\uf073'
  u+f0a3 x5   -> '\\leq '
  u+f03e x5   -> '\uf03e'
  u+f0ef x4   -> '(drop)'
  u+f0fc x3   -> '\uf0fc'
  u+f0fd x3   -> '\uf0fd'
  u+f0fe x3   -> '\uf0fe'
  u+f062 x3   -> '\uf062'
  u+f061 x2   -> '\uf061'
  u+f0b3 x2   -> '\\geq '
  u+f0e5 x1   -> '\\sum '
  u+f074 x1   -> '\uf074'


## 4. Реконструкция и парсинг формул СП 31

Собираем LaTeX геометрией (`tll.reconstruct_latex`), затем приводим к парсибельной форме
(`tll.to_parseable`): транслит кириллицы `cyr_map`, десятичная запятая → точка, `√` → `\sqrt`.
Считаем долю формул, которые `latex2sympy2` принимает.

In [ ]:
rows = []
with pdfplumber.open(PDF) as doc:
    for pno, page in enumerate(doc.pages, start=1):
        for num in tll._numbers(page.chars):
            disp = tll.reconstruct_latex(page, num)
            if not disp or ("=" not in disp and "≥" not in disp) or len(disp) < 4:
                continue
            parseable = tll.to_parseable(disp)
            try:
                latex2sympy(parseable)
                ok = True
            except Exception:
                ok = False
            rows.append((num, pno, disp, parseable, ok))

parsed = sum(r[4] for r in rows)
print(f"формул реконструировано: {len(rows)} | парсится: {parsed} ({parsed/len(rows):.0%})")
for num, pno, disp, parseable, ok in rows:
    print(f"  ({num:>2}) p{pno} {'OK ' if ok else 'ERR'} | {disp}  ->  {parseable}")

формул реконструировано: 13 | парсится: 9 (69%)
  ( 1) p12 OK  | Q_{ж}=\sum q_{ж}N_{ж}/1000,  ->  Q_{zh}=q_{zh}N_{zh}/1000
  ( 2) p13 OK  | Q^{сут.max}=KQ^{сут.m}.  ->  Q^{sutmax}=KQ^{sutm}.
  ( 3) p13 OK  | q^{ч.max}=KQ^{сут.max}24.  ->  q^{chmax}=KQ^{sutmax}24.
  ( 4) p13 ERR | _{K}^{ч.max}_{=}^{max}_{}^{max}_{.},  ->  _{K}^{chmax}_{=}^{max}_{}^{max}_{}
  ( 5) p39 ERR | Д_{щ}=К_{щ}(_{ек}^{к}−щ_{0})+1,  ->  D_{shch}=K_{shch}(_{ek}^{k}-shch_{0})+1
  ( 6) p42 OK  | h=\zeta\nu^{2}/2g,  ->  h=\zeta\nu^{2}/2g
  ( 7) p44 OK  | F_{в.о}=q\beta_{об}/3,6v_{p}N_{p},  ->  F_{vo}=q\beta_{ob}/3.6v_{p}N_{p}
  ( 8) p51 OK  | N_{ТМ}=√F_{ТМ}/2.  ->  N_{TM}=\sqrt{F_{TM}}/2
  ( 9) p51 OK  | v_{ф}=v_{н}N_{ф}/(N_{ф}−N_{1}),  ->  v_{f}=v_{n}N_{f}/(N_{f}-N_{1})
  (10) p54 OK  | Н_{ж}=Н_{з}а_{з}/100+0,3,  ->  N_{zh}=N_{z}a_{z}/100+0.3
  (10) p64 OK  | Д_{х}=0,7  ->  D_{kh}=0.7
  (11) p64 ERR | Д_{п}=  ->  D_{p}=
  (13) p85 ERR | PN≥^{ра}_{1}^{б}_{,4}^{\tau},  ->  PN\geq^{ra}_{1}^{b}_{,4}^{\tau}


## 5. Кириллические `id`: без коллизий с латиницей

Отдельный фикс: транслит `cyr_map` схлопывал кириллицу и латиницу в один `id`. Теперь
обозначению с кириллицей добавляется суффикс `_cyr` с сохранением регистра. Детальные числа
и миграция БД — в `nb05` §4 и `parser/cyrillic.py`.

In [ ]:
for latin, cyr in [("s", "с"), ("a", "а"), ("K", "К"), ("H", "Н")]:
    print(f"  лат {latin!r} -> {normalize_symbol(latin)}    кир {cyr!r} -> {normalize_symbol(cyr)}")

  лат 's' -> ('s', 's')    кир 'с' -> ('s_cyr', '\\text{с}')
  лат 'a' -> ('a', 'a')    кир 'а' -> ('a_cyr', '\\text{а}')
  лат 'K' -> ('K', 'K')    кир 'К' -> ('K_cyr', '\\text{К}')
  лат 'H' -> ('H', 'H')    кир 'Н' -> ('N_cyr', '\\text{Н}')


## 6. Выводы

- Текстовый слой **сохраняет кириллицу** (`h_ж`, `F_в.о`, `P_раб`); pix2text её калечил
  (`h_3arrom`) — это и был блокер.
- Геометрия `pdfplumber` (`size`/`top`) детерминированно даёт подстрочные/надстрочные индексы;
  PUA Symbol-шрифта закрывается таблицей; NFKC — математическими начертаниями.
- Итог на СП 31: **9/13 (69%)** формул парсятся (у pix2text было 10/21 = 48%), и парсятся
  **все формулы с кириллическими индексами**, чего pix2text не мог.
- Остаточные отказы — не про алфавит: дробь с горизонтальной чертой, химия `Fe^{2+}`,
  многоколоночная вёрстка. Это следующий шаг нормализации, а не кириллический блокер.

**Для продакшена:** расширить `[parser.pua_map]` до полной таблицы Symbol, включить шаг как
детерминированный OCR-free путь (pix2text остаётся для сканов). Владелец — Egor.